this is to follow the almgren_chris_model [link](https://github.com/viai957/Optimal-Portfolio-Transactions/blob/master/Almgren%20and%20Chriss%20Model.ipynb)

In [6]:
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

In [2]:
def get_kappa_tilde(
    tau: float,
    volatility: float,
    risk_penalty: float,
    permant_coeff_gamma: float,
    temp_impact_coeff_eta: float,
) -> float:
    eta_tilde = temp_impact_coeff_eta * (1 - permant_coeff_gamma * tau / (2 * temp_impact_coeff_eta))
    kappa_tilde = risk_penalty * (volatility**2) / eta_tilde
    return kappa_tilde


def get_kappa(
    tau: float,
    volatility: float,
    risk_penalty: float,
    permant_coeff_gamma: float,
    temp_impact_coeff_eta: float,
) -> float:
    kappa_tilde = get_kappa_tilde(tau, volatility, risk_penalty, permant_coeff_gamma, temp_impact_coeff_eta)
    kappa = 1 / tau * np.arccosh(tau * tau * 0.5 * kappa_tilde * kappa_tilde + 1)
    return kappa


def get_scheduling(
    X: float,
    N: int,
    tau: float,
    volatility: float,
    risk_penalty: float,
    permant_coeff_gamma: float,
    temp_impact_coeff_eta: float,
) -> float:
    T = N * tau
    t_j = tau * np.arange(0, N + 1)
    kappa = get_kappa(tau, volatility, risk_penalty, permant_coeff_gamma, temp_impact_coeff_eta)
    xj = np.sinh(kappa * (T - t_j)) / np.sinh(kappa * T) * X
    nj = 2 * np.sinh(0.5 * kappa * tau) / np.sinh(kappa * T) * np.cosh(kappa * (T - (t_j - 0.5 * tau))) * X
    return xj, nj

In [32]:
X = 100
volatility = 0.02
risk_penalty = 1e-2
permant_coeff_gamma = 1e-7
temp_impact_coeff_eta = 1e-5
N = 10
tau = 1.0

In [33]:
xj, nj = get_scheduling(X, N, tau, volatility, risk_penalty, permant_coeff_gamma, temp_impact_coeff_eta)
df = pd.DataFrame({"xj": xj, "nj": nj})
df.plot(y="nj", markers="o")

In [35]:
xj, nj = get_scheduling(
    X - 49.1137,
    N - 1,
    tau,
    volatility,
    risk_penalty,
    permant_coeff_gamma,
    temp_impact_coeff_eta,
)
df = pd.DataFrame({"xj": xj, "nj": nj})
df.plot(y="nj", markers="o")

The draw back is that the optimization due to the nonlinearity on the risk part, after the trade is done and redo the calculation will give completely different scheduling. which mean the execution is path dependent